# Лекция 4. Граф в памяти компьютера: четыре способа хранения

> Конспект четвёртого занятия курса «Алгоритмы и структуры данных» (ДПО).
> В лекции 3 дерево хранилось ссылками: вершина знала двух своих потомков.
> Граф устроен свободнее, ребро может соединить любые две вершины.
> Вопрос всей лекции: как хранить такую сеть, чтобы нужный вопрос о ней был дешёвым, а память не уходила впустую.
> Ответов четыре: матрица смежности, список рёбер, матрица инцидентности и список смежности.
> На десяти задачах контеста видно, какой вопрос какой структуре по силам.

**После лекции вы сможете:**

- различать рёбра и дуги, смежные и инцидентные, простой, плотный, разреженный и полный граф, граф и сеть;
- пронумеровать вершины и перевести номера из условия задачи в индексы Python;
- построить матрицу смежности, список рёбер, матрицу инцидентности и список смежности;
- посчитать память каждой структуры и объяснить, почему матрица смежности годится только для плотных графов;
- выбрать структуру под вопрос: есть ли ребро, кто соседи вершины, кто входит в вершину, какая у неё степень.

**Что нужно знать:** лекция 2: граф `G = {V, E}`, вершина, ребро, дерево, корень, предок, потомок. Лекция 3: вершина со ссылками на потомков. Оценки `O(n)` и `O(n²)`. Python: списки, списки списков, словари, `input()`.

**Время:** 35 минут на чтение. Около двух с половиной часов, если запускать код, вводить примеры и делать все предсказания. Удобная пауза после раздела 5. Вернувшись, перезапустите ячейки разделов 1 и 2: в них живут `n`, граф `edges` и дуги `arcs`, дальше лекция пользуется ими постоянно.

**Как читать.** Ячейки запускайте по порядку, сверху вниз. Блоки «Предскажите» и «Попробуйте сами» это барьеры: сначала отвечаете сами, потом открываете спойлер. Спойлеры обведены горизонтальными линиями. Решения задач контеста читают ввод через `input()`, как в Яндекс Контесте. Запустите ячейку и наберите строки примера из условия, по одной, каждую завершая Enter. Ожидаемый ответ записан комментарием в первой строке ячейки, а сам пример стоит над ячейкой блоком, строка к строке. Ячейки контеста перезаписывают `n`: если вводили свои данные, а не пример, перезапустите ячейку раздела 1. Остальные ячейки выполняются сами.

---

## 1. Граф в общем виде: вершины, рёбра, соседи

Слово «граф» вы встретили в лекции 2: это пара `G = {V, E}`, где `V` множество вершин, а `E` множество рёбер. Дерево из лекций 2 и 3 тоже граф, только особенный. В нём нет замкнутых маршрутов, и у каждой вершины, кроме корня, ровно один предок. Поэтому дерево хранилось просто. В лекции 2 куча лежала в списке, и предок находился арифметикой. В лекции 3 вершина помнила двух своих потомков.

Обычный граф таких ограничений не знает. Дороги между городами, связи в социальной сети, провода между серверами. Ребро может соединить любые две вершины, циклов сколько угодно, у вершины может быть тысяча соседей или ни одного. Такой граф приходится хранить иначе, и вопрос в том, как именно.

Вершины здесь нумеруются числами. Ребро записывают парой номеров: `(0, 2)` это ребро между вершинами `0` и `2`. Вот граф, который пройдёт через всю лекцию:

In [ ]:
n = 5                              # вершины 0, 1, 2, 3, 4
edges = [(0, 2), (1, 2), (1, 4)]   # три ребра

print("вершин:", n)
print("рёбер:", len(edges))
for a, b in edges:
    print(f"ребро ({a}, {b}) соединяет вершины {a} и {b}")

![Граф из пяти вершин и трёх рёбер](img/graph.png)

Два слова для отношений внутри графа легко перепутать, их важно различать сразу.

- Две вершины **смежны** (adjacent), если их соединяет ребро. Вершины `1` и `2` смежны, вершины `0` и `1` нет.
- Вершина и ребро **инцидентны** (incident), если вершина это один из концов ребра. Вершина `1` инцидентна рёбрам `(1, 2)` и `(1, 4)`, а вершина `2` инцидентна тому же ребру `(1, 2)` с другого конца.

Смежность связывает вершину с вершиной, инцидентность связывает вершину с ребром. От этих двух слов происходят названия двух матриц в разделах 4 и 7.

**Степень** вершины (degree), `deg`, это число инцидентных ей рёбер. У вершины `1` степень 2, у вершины `3` степень 0: она ни с кем не соединена. В простом графе, о котором речь в разделе 2, степень равна и числу смежных вершин.

**Предскажите.** Какие вершины смежны с вершиной `2`? Каким рёбрам она инцидентна? Какая у неё степень?

---

<details><summary>Ответ</summary>

Смежны `0` и `1`: рёбра `(0, 2)` и `(1, 2)` ведут к ним. Инцидентна вершина `2` этим же двум рёбрам. Степень 2.

Степень считается по инцидентным рёбрам. Здесь граф простой, поэтому ответ совпадает с числом смежных вершин. Если между двумя вершинами два параллельных ребра, степень вырастет на два, а смежная вершина останется одна.
</details>

---

## 2. Рёбра и дуги, петли и кратные рёбра, сеть

У ребра `(0, 2)` из раздела 1 нет направления: по нему можно пройти и из `0` в `2`, и обратно. Такой граф называют **неориентированным** (undirected). Бывает иначе. Улица с односторонним движением, подписка в соцсети, ссылка с одной страницы на другую. Здесь у связи есть начало и конец.

Граф, у которого связи направлены, называют **ориентированным** (directed). Его направленные рёбра называют **дугами** (arcs). Дуга `(4, 1)` ведёт из `4` в `1`, и обратного пути по ней нет. На рисунках дугу рисуют стрелкой.

Ещё два особых случая.

- **Петля** (loop): ребро из вершины в неё саму, `(3, 3)`.
- **Кратные рёбра** (multiple edges), их ещё называют параллельными: несколько рёбер между одной и той же парой вершин. Две разные дороги из одного города в другой.

Граф без петель и кратных рёбер называют **простым** (simple). Во многих задачах контеста граф простой, и это прямо сказано в условии.

Последний шаг. Сопоставьте каждой дуге `a` число `f(a)`: длину дороги, время в пути, цену перевозки. Буква `a` здесь от английского arc, это сама дуга. Граф, на дугах которого задана такая функция, называют **взвешенным** (weighted), а на занятии для него было ещё одно слово: **сеть** (network). Название темы занятия «Представление сетей в компьютере» ровно про это.

In [ ]:
# та же форма, но ориентированная и с весами: дуга (откуда, куда, цена)
arcs = [(0, 2, 7), (1, 2, 4), (4, 1, 3)]
for a, b, cost in arcs:
    print(f"дуга {a} -> {b}, f = {cost}")

Вершины в этих примерах уже пронумерованы. Откуда берутся номера, следующий раздел.

---

## 3. Индексация вершин

Вершины в жизни называются как угодно: «Москва», «сервер-7», «пользователь 815 042». Хранить граф удобнее, когда вершины это номера подряд: `0, 1, 2, ...`. Тогда номер вершины сразу становится индексом в списке. Присвоение номеров называют **индексацией вершин** (vertex indexing).

Индексацию делают словарём: название переводится в номер, а обратно помогает список.

In [ ]:
cities = ["Москва", "Тверь", "Клин", "Дубна", "Сергиев Посад"]
index = {name: i for i, name in enumerate(cities)}

print(index)
print("номер Клина:", index["Клин"])
print("вершина 3 это", cities[3])

В Яндекс Контесте вершины нумеруют с единицы: `1, 2, ..., n`. Списки Python начинаются с нуля. Поэтому все решения этой лекции переводят номер из условия строкой `i - 1`.

Граф из раздела 1 в условиях задач выглядит так: `1 3`, `2 3`, `2 5`. Это те же рёбра `(0, 2)`, `(1, 2)`, `(1, 4)`, только нумерация с единицы. Примеры задач B2, C2, E2 и F2 построены именно на нём.

### Где подведёт: забыть сдвинуть номер

In [ ]:
adj_rows = [[0] * 5 for _ in range(5)]
contest_edges = [(1, 3), (2, 3), (2, 5)]     # как в условии, с единицы

try:
    for i, j in contest_edges:
        adj_rows[i][j] = 1                   # забыли вычесть 1
except IndexError as error:
    print("упало на ребре", (i, j), ":", error)

Первые два ребра прошли молча и записались не в те клетки. Упало только третье, потому что вершины `5` в списке из пяти элементов нет. Опаснее всего как раз молчаливые ошибки: если бы вершины `5` в графе не было, программа отработала бы и выдала неверный ответ.

---

## 4. Матрица смежности

Первый способ хранения. Заведите таблицу `n × n`. В клетке `[i][j]` стоит 1, если вершины `i` и `j` смежны, и 0, если нет. Такую таблицу называют **матрицей смежности** (adjacency matrix). В Python это список из `n` списков.

In [ ]:
adj = [[0] * n for _ in range(n)]
for a, b in edges:
    adj[a][b] = 1
    adj[b][a] = 1          # граф неориентированный: ребро видно с обеих сторон

for row in adj:
    print(*row)

У неориентированного графа матрица симметрична: `adj[i][j] == adj[j][i]`. Каждое ребро записано дважды, над диагональю и под ней. На диагонали стоят петли: `adj[i][i] == 1` означает ребро из `i` в `i`.

Главная сила матрицы: вопрос «есть ли ребро между `i` и `j`» стоит один шаг. Нужно просто посмотреть в клетку. На занятии это назвали операцией `i-j`: здесь дефис значит «пара `i` и `j`», а не вычитание.

In [ ]:
print("1 и 2 смежны:", adj[1][2] == 1)
print("0 и 1 смежны:", adj[0][1] == 1)

У ориентированного графа симметрии нет. Строка `i` перечисляет дуги, которые выходят из `i`. Столбец `j` перечисляет дуги, которые входят в `j`. Проверьте на дугах из раздела 2.

In [ ]:
directed = [[0] * n for _ in range(n)]
for a, b, cost in arcs:
    directed[a][b] = 1     # только одна клетка: у дуги есть направление

for row in directed:
    print(*row)

v = 1
print(f"из {v} выходят дуги в:", [j for j in range(n) if directed[v][j] == 1], "(строка)")
print(f"в {v} входят дуги из:", [i for i in range(n) if directed[i][v] == 1], "(столбец)")

Исходящие смотрим по строке, входящие по столбцу. Цена обоих вопросов одинаковая: пройти `n` клеток.

Теперь три задачи контеста, которые на занятии решали через матрицу. Во всех трёх граф приходит сразу матрицей, её только читают построчно.

### Задача A2: петли

> **A2. Петли.** По заданной матрице смежности неориентированного графа определите, содержит ли он петли.
> **Ввод.** На вход программы поступает число `n` (`1 ≤ n ≤ 100`) — количество вершин графа, а затем `n` строк по `n` чисел, каждое из которых равно 0 или 1, — его матрица смежности.
> **Вывод.** Выведите «YES», если граф содержит петли, и «NO» в противном случае.
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5
1 1 1 1 0
1 0 1 1 1
1 1 0 1 1
1 1 1 1 1
0 1 1 1 0
```

Вывод:

```
YES
```

Петля это единица на диагонали. Смотреть нужно только клетки `adj[i][i]`.

In [ ]:
# введите 6 строк из примера A2; ответ: YES
n = int(input())
adj = []
for _ in range(n):
    line = list(map(int, input().split()))
    adj.append(line)

ans = 'NO'
for i in range(n):
    if adj[i][i] == 1:          # диагональ: ребро из i в i
        ans = 'YES'

print(ans)

### Задача B2: число рёбер

> **B2. Подсчет количества ребер неориентированного графа.** Простой неориентированный граф задан матрицей смежности. Найдите количество рёбер в графе.
> **Ввод.** На вход программы поступает число `n` (`1 ≤ n ≤ 100`) — количество вершин в графе, а затем `n` строк по `n` чисел, каждое из которых равно 0 или 1, — его матрица смежности.
> **Вывод.** Выведите одно число — количество рёбер заданного графа.
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5
0 0 1 0 0
0 0 1 0 1
1 1 0 0 0
0 0 0 0 0
0 1 0 0 0
```

Вывод:

```
3
```

Это матрица графа из раздела 1.

**Предскажите.** Сумма всех клеток этой матрицы равна 6, а рёбер три. Почему решение делит сумму на 2? Сработало бы деление, если бы в графе была петля?

---

<details><summary>Ответ</summary>

Граф неориентированный, поэтому каждое ребро `(i, j)` записано в матрице дважды: в клетке `[i][j]` и в клетке `[j][i]`. Сумма считает каждое ребро два раза.

С петлёй деление бы сломалось. Петля `(i, i)` занимает одну клетку на диагонали, и деление пополам превратило бы её в половину ребра. Поэтому в условии сказано «простой граф»: петель нет, и `// 2` честно.
</details>

---

In [ ]:
# введите 6 строк из примера B2; ответ: 3
n = int(input())
adj = []
for _ in range(n):
    line = list(map(int, input().split()))
    adj.append(line)
ans = 0
for i in range(n):
    for j in range(n):
        ans += adj[i][j]
print(ans // 2)                 # каждое ребро посчитано дважды

### Задача C2: от матрицы к списку рёбер

> **C2. От матрицы смежности к списку ребер (неориентированный).** Простой неориентированный граф задан матрицей смежности, выведите его представление в виде списка рёбер.
> **Ввод.** Входные данные включают число `n` (`1 ≤ n ≤ 100`) — количество вершин в графе, а затем `n` строк по `n` чисел, каждое из которых равно 0 или 1, — его матрицу смежности.
> **Вывод.** Выведите список рёбер заданного графа (в любом порядке).
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5
0 0 1 0 0
0 0 1 0 1
1 1 0 0 0
0 0 0 0 0
0 1 0 0 0
```

Вывод:

```
1 3
2 3
2 5
```

Та же проблема двойной записи, но решена иначе. Вместо деления решение смотрит только клетки над диагональю: `j` начинается с `i + 1`. Каждое ребро там встречается ровно один раз.

In [ ]:
# введите 6 строк из примера C2 (он такой же, как в B2); ответ: 1 3, 2 3, 2 5 по строкам
n = int(input())
adj = []
for _ in range(n):
    line = list(map(int, input().split()))
    adj.append(line)

for i in range(n):
    for j in range(i + 1, n):          # только над диагональю
        if adj[i][j] == 1:
            print(i + 1, j + 1)        # обратно к нумерации с единицы

Вывод C2 это уже второй способ хранения графа. О нём следующий раздел.

---

## 5. Список рёбер

Второй способ самый прямой. Запишите рёбра одно за другим, как они есть. Это **список рёбер** (edge list). Именно так граф приходит во входных данных большинства задач: число вершин, число рёбер и затем пары.

Для сети к каждой дуге добавляют вес. На занятии это разложили в три массива одинаковой длины: `From` откуда, `To` куда, `C` стоимость. Дуга номер `k` это `From[k] → To[k]` с ценой `C[k]`. Порядок рёбер никакой, их складывают навалом.

In [ ]:
From = [a for a, b, cost in arcs]
To = [b for a, b, cost in arcs]
C = [cost for a, b, cost in arcs]

print("From:", From)
print("To:  ", To)
print("C:   ", C)
for k in range(len(From)):
    print(f"дуга {k}: {From[k]} -> {To[k]}, цена {C[k]}")

Проблема списка рёбер видна сразу. Спросите его о вершине: кто соседи вершины `1`? Где её рёбра, неизвестно. Придётся просмотреть все. Любой вопрос о конкретной вершине стоит `O(|E|)`, где `|E|` это число рёбер.

In [ ]:
v = 1
steps = 0
found = []
for a, b in edges:
    steps += 1
    if a == v:
        found.append(b)
    elif b == v:
        found.append(a)
print(f"соседи {v}: {found}, просмотрено рёбер: {steps} из {len(edges)}")

Из-за этого в задачах D2, E2, F2 и G2 список рёбер почти сразу перекладывают в матрицу. Список нужен, чтобы прочитать граф. Отвечать на вопросы удобнее матрице.

### Задача D2: от списка рёбер к матрице

> **D2. От списка ребер к матрице смежности (ориентированный).** Простой ориентированный граф задан списком рёбер, выведите его представление в виде матрицы смежности.
> **Ввод.** На вход программы поступают числа `n` (`1 ≤ n ≤ 100`) — количество вершин в графе и `m` (`1 ≤ m ≤ n(n − 1)`) — количество рёбер. Затем следует `m` пар чисел — рёбра графа.
> **Вывод.** Выведите матрицу смежности заданного графа.
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5 3
1 3
2 3
5 2
```

Вывод:

```
0 0 1 0 0
0 0 1 0 0
0 0 0 0 0
0 0 0 0 0
0 1 0 0 0
```

Это дуги из раздела 2. Граф ориентированный, поэтому клетка заполняется одна.

In [ ]:
# введите 4 строки из примера D2; ответ: матрица из 5 строк, как в условии
n, m = map(int, input().split())
adj = [[0 for j in range(n)] for i in range(n)]
for _ in range(m):
    i, j = map(int, input().split())
    adj[i - 1][j - 1] = 1              # дуга i -> j: одна клетка
for line in adj:
    print(*line, sep=' ')

### Задача E2: параллельные рёбра

> **E2. Проверка на наличие параллельных ребер (неориентированный).** Неориентированный граф задан списком рёбер. Проверьте, содержит ли он параллельные рёбра.
> **Ввод.** Сначала вводятся числа `n` (`1 ≤ n ≤ 100`) — количество вершин в графе и `m` (`1 ≤ m ≤ 10 000`) — количество рёбер. Затем следует `m` пар чисел — рёбра графа.
> **Вывод.** Выведите «YES», если граф содержит параллельные рёбра, и «NO» в противном случае.
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5 3
1 3
2 3
2 5
```

Вывод:

```
NO
```

Здесь видно, зачем перекладывать список в матрицу. Для каждого нового ребра нужно спросить: такое уже было? В списке рёбер это просмотр всех прочитанных, `O(|E|)` на каждое. При 10 000 рёбрах выходит около 50 миллионов сравнений. В матрице это одна клетка.

In [ ]:
# введите 4 строки из примера E2; ответ: NO
n, m = map(int, input().split())
adj = [[0 for j in range(n)] for i in range(n)]
ans = 'NO'
for _ in range(m):
    i, j = map(int, input().split())
    if adj[i-1][j-1] == 1:             # такое ребро уже встречалось
        ans = "YES"
    adj[i-1][j-1] = 1
    adj[j-1][i-1] = 1
print(ans)

Обратите внимание: проверяется одна клетка `[i-1][j-1]`, а записываются две. Ребро `2 3` и ребро `3 2` в неориентированном графе одно и то же. Симметричная запись ловит и такой повтор.

### Задача F2: степени вершин

> **F2. Степени вершин по спискам ребер.** Неориентированный граф задан списком рёбер. Найдите степени всех вершин графа.
> **Ввод.** Сначала вводятся числа `n` (`1 ≤ n ≤ 100`) — количество вершин в графе и `m` (`1 ≤ m ≤ n(n − 1)/2`) — количество рёбер. Затем следует `m` пар чисел — рёбра графа.
> **Вывод.** Выведите `n` чисел — степени вершин графа.
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5 3
1 3
2 3
2 5
```

Вывод:

```
1
2
2
0
1
```

Степень вершины это сумма её строки в матрице. Решение с занятия прибавляет по единице, а не ставит единицу. Так оно считает и кратные рёбра, если они попадутся.

In [ ]:
# введите 4 строки из примера F2; ответ: 1, 2, 2, 0, 1 по строкам
n, m = map(int, input().split())
adj = [[0 for j in range(n)] for i in range(n)]
for _ in range(m):
    i, j = map(int, input().split())
    adj[i-1][j-1] += 1
    adj[j-1][i-1] += 1
print(*[sum(i) for i in adj], sep='\n')   # степень = сумма строки

### Где подведёт: одна клетка вместо двух

Для неориентированного графа ребро надо записать в обе клетки. Посмотрите, что будет, если записать одну.

In [ ]:
half = [[0] * n for _ in range(n)]
for a, b in edges:
    half[a][b] += 1                    # забыли half[b][a]

print("степени по одной клетке:", [sum(row) for row in half])
print("верные степени:         ", [sum(v in edge for edge in edges) for v in range(n)])

Ошибка молчаливая: программа ничего не роняет и печатает правдоподобные числа. Вершины `2` и `4` потеряли свои рёбра. В парах `(0, 2)`, `(1, 2)`, `(1, 4)` они стоят вторыми, и их строки остались пустыми. Так выглядит неправильный ответ на тесте, где вы ждали правильный.

---

## 6. Сколько стоит матрица: плотный, разреженный и полный граф

Матрица отвечает на вопрос `i-j` за один шаг. Платит она памятью: `n × n` клеток всегда, сколько бы рёбер ни было.

На занятии это посчитали на сети из 4000 вершин и 16 000 дуг.

In [ ]:
def num(x):
    """Число с пробелами между разрядами: 16 000 000."""
    return f"{x:,}".replace(",", " ")


V, E = 4000, 16_000

matrix_cells = V * V
print(f"матрица смежности: {num(matrix_cells)} клеток")
print(f"из них с единицами: {num(E)}, то есть {E / matrix_cells:.1%}")
print(f"нулей: {1 - E / matrix_cells:.1%}")

edge_list_cells = 3 * E            # From, To, C
print(f"список рёбер с ценами: {num(edge_list_cells)} клеток")
print(f"отношение матрицы к списку рёбер: {matrix_cells // edge_list_cells}")

Матрица тратит 99,9% памяти на хранение нулей: на сведения о том, чего в графе нет.

Граф, у которого рёбер порядка `|V|²`, называют **плотным** (dense). Для него матрица оправдана: клеток того же порядка, что рёбер, и заметная их доля заполнена. Граф, у которого рёбер намного меньше `|V|²`, называют **разреженным** (sparse). Сеть из примера разреженная: 16 000 дуг против 16 миллионов возможных. Настоящие сети почти всегда разреженные: у города несколько дорог, а не дорога в каждый другой город.

Правило с занятия: матрицу смежности берут, когда `|E|` сравнимо с `|V|²`. В остальных случаях она крайне неэффективна.

На занятии плотный граф описали словами «из каждой вершины в каждую идёт». Точнее это определение **полного** графа (complete graph). Полный граф это крайний случай плотного: рёбер максимально возможное число. В неориентированном простом графе это `|V| · (|V| − 1) / 2` рёбер, в ориентированном `|V| · (|V| − 1)` дуг. Плотный граф до полного не дотягивает, но рёбер у него того же порядка.

In [ ]:
for V in (5, 100, 4000):
    print(f"|V| = {V}: полный неориентированный {num(V * (V - 1) // 2)} рёбер, "
          f"полный ориентированный {num(V * (V - 1))} дуг")

Проверить, полный ли граф, это как раз вопрос для матрицы. Две задачи контеста.

### Задача G2: полный граф

> **G2. Полный граф.** Неориентированный граф с кратными рёбрами называется **полным**, если любая пара его различных вершин соединена хотя бы одним ребром. Для заданного списком рёбер графа проверьте, является ли он полным.
> **Ввод.** Сначала вводятся числа `n` (`1 ≤ n ≤ 100`) — количество вершин в графе и `m` (`1 ≤ m ≤ n(n − 1)/2`) — количество рёбер. Затем следует `m` пар чисел — рёбра графа.
> **Вывод.** Выведите «YES», если граф является полным, и «NO» в противном случае.
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5 18
1 2
1 3
1 3
1 4
1 4
1 4
1 5
1 5
2 3
2 4
2 4
2 5
3 4
3 4
3 4
3 5
3 5
4 5
```

Вывод:

```
YES
```

Кратные рёбра здесь не мешают. Решение ставит в клетку 1, а не прибавляет, поэтому повторы ложатся в ту же единицу. Из-за кратных рёбер `m` в примере равно 18, хотя в условии написано `m ≤ n(n − 1)/2`, то есть не больше 10. Ограничение в условии кратные рёбра не учитывает, решение от него не зависит.

**Предскажите.** Перед сравнением решение заполняет диагональ единицами и сравнивает сумму матрицы с `n * n`. Зачем диагональ?

---

<details><summary>Ответ</summary>

В полном графе без петель заполнены все клетки, кроме диагонали: сумма равна `n * n − n`. Решение дописывает диагональ, и тогда полный граф это просто матрица, где единица стоит везде. Сравнивать с `n * n` проще, чем помнить про вычитание.

Заодно это защищает от петель во входных данных. Если в графе есть петля, её единица на диагонали уже стоит, и заполнение ничего не добавит.
</details>

---

In [ ]:
# введите 19 строк из примера G2; ответ: YES
n, m = map(int, input().split())
adj = [[0 for j in range(n)] for i in range(n)]
for _ in range(m):
    i, j = map(int, input().split())
    adj[i - 1][j - 1] = 1
    adj[j - 1][i - 1] = 1
for i in range(n):
    adj[i][i] = 1                      # диагональ считаем заполненной
print("YES" if sum(map(sum, adj)) == n * n else "NO")

### Задача H2: полуполный граф

> **H2. Полуполный граф.** Ориентированный граф называется **полуполным**, если между любой парой его различных вершин есть хотя бы одно ребро. Для заданного списком рёбер графа проверьте, является ли он полуполным.
> **Ввод.** Сначала вводятся числа `n` (`1 ≤ n ≤ 100`) — количество вершин в графе и `m` (`1 ≤ m ≤ n(n − 1)`) — количество рёбер. Затем следует `m` пар чисел — рёбра графа.
> **Вывод.** Выведите «YES», если граф является полуполным, и «NO» в противном случае.
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5 10
1 2
1 3
1 5
2 3
2 5
3 2
4 1
4 3
4 5
5 3
```

Вывод:

```
NO
```

Такой граф называют **полуполным** (semicomplete). Чем отличается от G2. Граф ориентированный, и для пары `i, j` хватает одной дуги в любую сторону. Значит, для каждой пары нужно посмотреть две клетки: `[i][j]` и `[j][i]`. Если обе нули, граф не полуполный. В примере так у пары `2` и `4`. Решение перебирает все пары и каждую проверяет за один шаг: это вопрос как раз для матрицы.

In [ ]:
# введите 11 строк из примера H2; ответ: NO
n, m = map(int, input().split())
adj = [[0 for j in range(n)] for i in range(n)]
for _ in range(m):
    i, j = map(int, input().split())
    adj[i - 1][j - 1] = 1
ans = 'YES'
for i in range(n):
    for j in range(i + 1, n):
        if adj[i][j] == 0 and adj[j][i] == 0:   # ни туда, ни обратно
            ans = 'NO'
print(ans)

---

## 7. Матрица инцидентности

Матрица смежности держит в клетке пару вершин. А если сделать столбцами сами рёбра? Получится третий способ хранения. Строки это вершины, столбцы это рёбра. В клетке `[v][k]` стоит 1, если вершина `v` инцидентна ребру `k`. Такую таблицу называют **матрицей инцидентности** (incidence matrix). В каждом столбце ровно две единицы, два конца ребра. Исключение петля: у неё конец один.

In [ ]:
incidence = [[0] * len(edges) for _ in range(n)]
for k, (a, b) in enumerate(edges):
    incidence[a][k] = 1
    incidence[b][k] = 1

print("       рёбра:", *[f"{a}-{b}" for a, b in edges])
for v in range(n):
    print(f"вершина {v}:  ", *[f"{x:>3}" for x in incidence[v]])

Для ориентированного графа в одной из договорённостей в клетку ставят `+1` у начала дуги и `−1` у конца, так направление не теряется.

На занятии про эту структуру сказали коротко: ещё хуже. Посчитайте её для той же сети из 4000 вершин и 16 000 дуг.

**Предскажите.** Сколько клеток займёт матрица инцидентности этой сети? Сколько из них будет не нулями?

---

<details><summary>Ответ</summary>

`4000 · 16 000 = 64 000 000` клеток, вчетверо больше матрицы смежности. Не нулей всего `2 · 16 000 = 32 000`, по два в каждом столбце: 0,05%.

И ответить быстро ей тоже нечем. Чтобы узнать, соединены ли `i` и `j`, надо пройти по строке `i` через все `|E|` столбцов и в каждом с единицей проверить строку `j`. Это `O(|E|)`, как у списка рёбер, только памяти больше примерно в полторы тысячи раз.
</details>

---

Матрица инцидентности нужна в теории графов. Через неё, например, записывают задачу о потоке в сети как систему уравнений: в каждой вершине сколько втекло, столько и вытекло. Для хранения графа в программе её почти не используют.

---

## 8. Список смежности

Итак, матрица быстро отвечает, но тратит `|V|²` памяти. Список рёбер экономный, но на каждый вопрос о вершине просматривает все рёбра. Нужна структура, где каждая вершина сразу знает своих соседей и не хранит нулей.

Вы её уже видели. В лекции 3 вершина дерева хранила ссылки на двух потомков: `left` и `right`. Сделайте то же для графа, только вместо двух полей список произвольной длины. И одно отличие: в дереве вершина не помнила предка, а здесь в список попадают все соседи. Для каждой вершины храните список её соседей. Всё вместе называют **списком смежности** (adjacency list).

In [ ]:
neighbours = [[] for _ in range(n)]
for a, b in edges:
    neighbours[a].append(b)
    neighbours[b].append(a)        # неориентированный: сосед в обе стороны

for v in range(n):
    print(f"{v}: {neighbours[v]}")

Вопрос «кто соседи вершины `v`» теперь стоит ровно столько, сколько у неё соседей: `O(deg v)`. Не `n` клеток строки и не `|E|` рёбер списка, а только нужное.

Два других вопроса. Степень вершины это длина её списка, `len(neighbours[v])`, один шаг. В матрице для этого пришлось бы сложить строку, `n` клеток. Проверка пары `i-j` здесь дороже, чем в матрице: надо пройти список `i` и поискать в нём `j`, это `O(deg i)`.

**Предскажите.** В неориентированном графе сложите длины всех списков соседей. Чему равна сумма, если рёбер `|E|`?

---

<details><summary>Ответ</summary>

`2 · |E|`. Каждое ребро `(a, b)` попадает в два списка: `b` в список `a` и `a` в список `b`. В графе из раздела 1 три ребра и сумма длин `1 + 2 + 2 + 0 + 1 = 6`.

Длина списка вершины это её степень. Значит, сумма степеней всех вершин равна удвоенному числу рёбер. Это тот же факт, из-за которого в B2 сумму матрицы делили на 2. Его называют леммой о рукопожатиях (handshaking lemma): в каждом рукопожатии участвуют две руки.
</details>

---

Память списка смежности: `|V|` списков и в них `|E|` записей для ориентированного графа или `2 · |E|` для неориентированного. Оценка `O(|V| + |E|)`. Сравните для сети с занятия.

In [ ]:
import random

V, E = 4000, 16_000
random.seed(4)
network = [[] for _ in range(V)]
added = 0
while added < E:
    a, b = random.randrange(V), random.randrange(V)
    if a != b and b not in network[a]:
        network[a].append(b)            # ориентированная сеть: дуга a -> b
        added += 1

adjacency_cells = V + sum(len(x) for x in network)
print(f"матрица смежности:          {num(V * V):>10} клеток")
print(f"список рёбер (From, To, C): {num(3 * E):>10} клеток")
print(f"список смежности:           {num(adjacency_cells):>10} клеток (|V| + |E|)")

v = 0
print(f"соседи вершины {v}: {network[v]}")
print("чтобы их найти, нужно просмотреть:")
print(f"  в матрице смежности клеток строки: {V}")
print(f"  в списке рёбер записей: {num(E)}")
print(f"  в списке смежности записей: {len(network[v])}")

Если у дуг есть цена, в список кладут пары `(сосед, цена)`. Записей столько же, клеток вдвое больше.

In [ ]:
weighted = [[] for _ in range(n)]
for a, b, cost in arcs:
    weighted[a].append((b, cost))
print(weighted)

### Список по исходящим и по входящим

У ориентированного графа список смежности можно построить двумя способами. По исходящим дугам: в списке `v` те, куда ведут дуги из `v`. По входящим: в списке `v` те, откуда дуги приходят в `v`. На занятии это и значило «построить на входящих».

От выбора зависит, какой вопрос дешёвый. Список по исходящим отвечает «куда можно пойти из `v`» за столько шагов, сколько дуг выходит из `v`. А на вопрос «кто ведёт в `v`» ему придётся просмотреть все списки, то есть `O(|V| + |E|)`. Если нужны оба вопроса, строят оба списка.

In [ ]:
out_list = [[] for _ in range(n)]
in_list = [[] for _ in range(n)]
for a, b, cost in arcs:
    out_list[a].append(b)
    in_list[b].append(a)

v = 1
print(f"из {v} ведут дуги в: {out_list[v]}")
print(f"в {v} ведут дуги из: {in_list[v]}")

Это та же пара, что строка и столбец в матрице, только каждый ответ стоит столько, сколько дуг у `v`, а не `n` клеток.

### Задача F2 списком смежности

Задачу F2 можно решить без матрицы. Степень вершины это длина её списка соседей.

In [ ]:
# введите 4 строки из примера F2; ответ: 1, 2, 2, 0, 1 по строкам
n, m = map(int, input().split())
neighbours = [[] for i in range(n)]
for _ in range(m):
    i, j = map(int, input().split())
    neighbours[i - 1].append(j - 1)
    neighbours[j - 1].append(i - 1)
print(*[len(v) for v in neighbours], sep='\n')

При `n ≤ 100` разницы в скорости не заметно. Разница в памяти: матрица всегда `n · n` клеток, при `n = 100` это 10 000, а список ровно `n + 2 · m`. Для неориентированной сети из 4000 вершин и 16 000 рёбер это 16 миллионов против `4000 + 32 000 = 36 000`.

---

## 9. Структура под вопрос: транзитивность

Последние две задачи показывают, что выбирать структуру приходится под вопрос. Иногда нужны сразу две.

Граф называют **транзитивным**, если для различных вершин `u` и `w` из того, что `u` смежна с `v`, а `v` смежна с `w`, всегда следует, что `u` смежна с `w`. Слово взято из математики, так называют отношение (transitive relation). Пример из жизни: «работают в одном отделе». Если Анна с Борисом в одном отделе, а Борис с Верой, то и Анна с Верой. Транзитивный неориентированный граф так и выглядит: он распадается на группы, где каждый связан с каждым.

Почему `u` и `w` различные. Иначе из `u–v` и `v–u` следовало бы ребро `u–u`, то есть петля, и любой граф с ребром оказался бы нетранзитивным.

### Задача I2: транзитивность неориентированного графа

> **I2. Транзитивность неориентированного графа.** Напомним, что граф называется **транзитивным**, если всегда из того, что вершины `u` и `v` соединены рёбром и вершины `v` и `w` соединены рёбром, следует, что вершины `u` и `w` соединены рёбром. Проверьте, что заданный неориентированный граф является транзитивным.
> **Ввод.** Сначала вводятся числа `n` (`1 ≤ n ≤ 100`) — количество вершин в графе и `m` (`1 ≤ m ≤ n(n − 1)/2`) — количество рёбер. Затем следует `m` пар чисел — рёбра графа.
> **Вывод.** Выведите «YES», если граф является транзитивным, и «NO» в противном случае.
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5 1
2 5
```

Вывод:

```
YES
```

**Попробуйте сами, прежде чем читать дальше.** В проверке два действия. Для каждой вершины `v` перебрать все пары её соседей `u` и `w`. Для каждой пары спросить, смежны ли `u` и `w`. Какая структура лучше для первого действия, а какая для второго?

---

<details><summary>Ответ</summary>

Перебор соседей удобен в списке смежности: он отдаёт соседей `v` за `O(deg v)`, без пустых клеток. Вопрос «смежны ли `u` и `w`» удобен в матрице: одна клетка.

Поэтому решение строит обе структуры из одного списка рёбер. Памяти при `n ≤ 100` на это хватает с запасом, а каждая часть проверки получает свою быструю структуру.

Матрица помогает и при чтении. Условие не запрещает повторять ребро, а повторы в списке соседей раздули бы перебор пар. Поэтому ребро кладётся в список, только если в матрице его ещё нет.
</details>

---

In [ ]:
# введите 2 строки из примера I2; ответ: YES
n, m = map(int, input().split())
adj = [[0 for j in range(n)] for i in range(n)]
neighbours = [[] for i in range(n)]
for _ in range(m):
    i, j = map(int, input().split())
    if adj[i - 1][j - 1] == 0:             # такого ребра ещё не было
        adj[i - 1][j - 1] = 1
        adj[j - 1][i - 1] = 1
        neighbours[i - 1].append(j - 1)
        neighbours[j - 1].append(i - 1)
ans = 'YES'
for v in range(n):
    for u in neighbours[v]:                # соседи v: из списка смежности
        for w in neighbours[v]:
            if u != w and adj[u][w] == 0:  # смежны ли u и w: из матрицы
                ans = 'NO'
print(ans)

### Задача J2: транзитивность ориентированного графа

> **J2. Транзитивность ориентированного графа.** Напомним, что ориентированный граф называется **транзитивным**, если для любых трёх различных вершин `u`, `v` и `w` из того, что из `u` в вершину `v` ведёт ребро и из вершины `v` в вершину `w` ведёт ребро, следует, что из вершины `u` в вершину `w` ведёт ребро. Проверьте, что заданный ориентированный граф является транзитивным.
> **Ввод.** Сначала вводится число `n` (`1 ≤ n ≤ 100`) — количество вершин в графе, а затем `n` строк по `n` чисел, каждое из которых равно 0 или 1, — его матрица смежности.
> **Вывод.** Выведите «YES», если граф является транзитивным, и «NO» в противном случае.
> **Ограничения.** 1 секунда, 64 МБ.

Пример из условия. Ввод:

```
5
0 0 0 0 0
0 0 0 0 0
0 0 0 0 0
0 0 0 0 0
0 0 0 0 0
```

Вывод:

```
YES
```

Чем отличается от I2. Граф приходит матрицей, и дуги направлены: `u → v` и `v → w` должны дать `u → w`. Соседей здесь перебирает строка матрицы. Цепочка `u → v → w` это строка `u`, а затем строка `v`.

In [ ]:
# введите 6 строк из примера J2; ответ: YES
n = int(input())
adj = []
for _ in range(n):
    line = list(map(int, input().split()))
    adj.append(line)
ans = 'YES'
for u in range(n):
    for v in range(n):
        if u != v and adj[u][v] == 1:              # дуга u -> v
            for w in range(n):
                if w != u and w != v and adj[v][w] == 1 and adj[u][w] == 0:
                    ans = 'NO'                     # есть u -> v -> w, нет u -> w
print(ans)

Пример в условии пустой: граф без дуг транзитивен, потому что нарушить правило нечем. Проверьте на графе с настоящей цепочкой, не вводя ничего с клавиатуры.

In [ ]:
def transitive(adj):
    n = len(adj)
    for u in range(n):
        for v in range(n):
            if u != v and adj[u][v] == 1:
                for w in range(n):
                    if w != u and w != v and adj[v][w] == 1 and adj[u][w] == 0:
                        return False
    return True


chain = [[0, 1, 0],
         [0, 0, 1],
         [0, 0, 0]]            # 0 -> 1 -> 2, а дуги 0 -> 2 нет
print("цепочка без замыкания:", transitive(chain))
chain[0][2] = 1
print("после дуги 0 -> 2:    ", transitive(chain))

Четыре структуры и десять задач позади. Осталось свести всё в одну таблицу.

---

## 10. Четыре структуры в одной таблице

В таблице `V` это число вершин `|V|`, `E` число рёбер `|E|`. Степень вершины матрица считает за `O(V)`, список смежности за один шаг.

| Структура | Память | Есть ли ребро `i-j` | Соседи вершины `v` | Когда брать |
|---|---|---|---|---|
| Матрица смежности | `V²` | `O(1)` | `O(V)`: строка или столбец | плотный граф, частый вопрос `i-j` |
| Список рёбер | `2 · E`, с ценой `3 · E` | `O(E)` | `O(E)` | чтение и запись графа, обход всех рёбер подряд |
| Матрица инцидентности | `V · E` | `O(E)` | `O(E + deg v · V)` | для хранения в программе почти не берут |
| Список смежности | `O(V + E)` | `O(deg i)`: поиск `j` в списке `i` | `O(deg v)` по исходящим; для входящих нужен второй список | разреженный граф, перебор соседей |

![Один граф в трёх структурах](img/views.png)

Для сети с занятия, 4000 вершин и 16 000 дуг: матрица смежности 16 миллионов клеток, матрица инцидентности 64 миллиона, список рёбер с ценами 48 тысяч, список смежности 20 тысяч.

На рисунке граф из раздела 1 в трёх структурах. Матрицы инцидентности на нём нет: её вывод есть в разделе 7.

---

## 11. Проверьте себя

Ответьте без кода, потом сверьтесь.

1. Чем смежность отличается от инцидентности? Приведите пример для графа из раздела 1.
2. В матрице смежности ориентированного графа где искать дуги, которые входят в вершину `5`? Нумерация вершин с нуля.
3. Граф на 1000 вершин, у каждой примерно 10 соседей. Плотный он или разреженный? Сколько клеток займёт матрица смежности и сколько записей список смежности?
4. Почему в B2 сумму матрицы делят на 2, а в D2 нет?
5. В задаче I2 решение строит и матрицу, и список смежности. Для чего каждая?

---

<details><summary>Ответы</summary>

1. Смежность это отношение двух вершин: они соединены ребром. Инцидентность это отношение вершины и ребра: вершина это один из концов ребра. В графе из раздела 1 вершины `1` и `4` смежны, а вершина `1` инцидентна ребру `(1, 4)`.
2. В столбце `5`. Строка перечисляет исходящие дуги, столбец входящие.
3. Разреженный: рёбер около `1000 · 10 / 2 = 5000`, а возможных почти полмиллиона. Матрица займёт `1000 · 1000 = 1 000 000` клеток. Список смежности `1000` списков и `2 · 5000 = 10 000` записей.
4. В B2 граф неориентированный, и каждое ребро записано в матрице дважды. В D2 граф ориентированный, и каждая дуга занимает одну клетку.
5. Список смежности перебирает соседей вершины `v` без пустых клеток. Матрица за один шаг отвечает, смежны ли два соседа между собой.
</details>

---

## 12. Итог

- Граф хранят четырьмя способами. Матрица смежности, список рёбер, матрица инцидентности, список смежности. Выбирают по двум вопросам: сколько памяти и какой вопрос к графу будет частым.
- Матрица смежности отвечает на вопрос `i-j` за один шаг, но всегда занимает `|V|²`. Она оправдана для плотного графа, где рёбер порядка `|V|²`. Настоящие сети разреженные. В сети с занятия матрица на 99,9% хранит нули.
- Список рёбер экономный, в нём граф приходит на вход, но любой вопрос о вершине стоит `O(|E|)`. Матрица инцидентности хуже обоих.
- Список смежности занимает `O(|V| + |E|)` и отдаёт соседей вершины за `O(deg v)`. Для разреженной сети это лучший выбор, а если нужны и соседи, и проверка пары, строят две структуры, как в I2.

**Где встретится дальше.** Список смежности это рабочая форма графа почти для всех алгоритмов на графах: обходы, поиск кратчайших путей, поиск компонент. Слова «вершина», «дуга», «вес», «степень» и оценка `O(|V| + |E|)` понадобятся сразу.

**Что повторить через два-три дня.** Возьмите ориентированный граф из пяти вершин с дугами `0 → 1`, `1 → 2`, `2 → 0`, `3 → 4`. Нарисуйте его, запишите матрицу смежности, список рёбер и список смежности по исходящим. Ответьте по каждой структуре, кто входит в вершину `0`, и посчитайте, сколько шагов на это ушло.